In [3]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
import os

In [2]:
# Load labels and pockets
root = '.'
PATH_TO_PROBS = os.path.join(root, "..", "processed", "unidock_docking", "inference_probs")
labels = np.array(pickle.load(open(os.path.join(PATH_TO_PROBS, "success_mols.pkl"), "rb")))
pockets = [i.replace("_bin_01.npz", "") for i in sorted(os.listdir(PATH_TO_PROBS)) if i != "success_mols.pkl"]

print(f"Number of pockets: {len(pockets)}")
print(f"Number of unique molecules: {len(set(labels))}")

Number of pockets: 276
Number of unique molecules: 9556875


In [4]:
# Load probabilities
print("Loading probabilities...")
pocket_to_all_probs = {}
for c, pocket in tqdm(enumerate(pockets)):
    probs = np.load(os.path.join(PATH_TO_PROBS, f"{pocket}_bin_01.npz"))["arr_0"]
    pocket_to_all_probs[pocket] = probs

Loading probabilities...


276it [01:02,  4.42it/s]


In [ ]:
# Prepare matrix
M = np.array([pocket_to_all_probs[p] for p in pockets])
print(f"Matrix shape before transposition: {M.shape}")
del pocket_to_all_probs
M = M.T
print(f"Matrix shape: {M.shape}")

print("Normalizing by columns...")

# Calculate means
means_1 = np.mean(M, axis=0, keepdims=True)

# Calculate stds
stds_1 = np.std(M, axis=0, keepdims=True)

# Z-score (1)
M -= means_1
M /= stds_1

print("Normalizing by rows...")

# Calculate means
means_2 = np.mean(M, axis=1, keepdims=True)

# Calculate stds
stds_2 = np.std(M, axis=1, keepdims=True)

# Z-score (2)
M -= means_2
M /= stds_2